<h1 style="font-family:verdana;"> <center>📚Tabular-Playground-Series-September 2022 EDA + Training Different Models</center> </h1>
<p><center style="color:#159364; font-family:cursive;">Thanks for visiting my notebook </center></p>

***

<center><img src='https://media2.giphy.com/media/hAieQ20Ph6xJPnVqLr/giphy.webp?cid=ecf05e47yuj5a5zofn9vl9wocpd2i5801opstasr7drbgqml&rid=giphy.webp&ct=s' 
     height=30px width=160px /></center>
     
## ⚠️Warning
<div class="alert alert-block alert-info" style="font-size:14px; font-family:verdana;">
    📌 The whole content might take some time to fully load due to the plotly library and graphs. Please wait atleast 10-15 seconds or reload the page if it takes too long. Thanks! 😊
</div>

## 🔬Overview
<p style="font-size:15px; font-family:verdana; line-height: 1.7em">
The competing Kaggle merchandise stores we saw in January's Tabular Playground are at it again. This time, they're selling books!</p><br>

<p style="font-size:15px; font-family:verdana; line-height: 1.7em">
The task for this month's competitions is a bit more complicated. Not only are there six countries and four books to forecast, but you're being asked to forecast sales during the tumultuous year 2021. Can you use your data science skills to predict book sales when conditions are far from the ordinary? </p><br>

<p style="font-size:15px; font-family:verdana; line-height: 1.7em">
In this notebook, we will be exploring the dataset and visualize the relationships of every column through graphs are representation.We will then preprocess the data and perform some feature engineering in order to be able to fit the train data into different machine learning models. After that, we will evaluate the models using different metrics to determine which model will perform best on the current dataset. </p>

## 💡Inspiration
<p style="font-size:15px; font-family:verdana; line-height: 1.7em">
This notebook is heavily inspired by the this 
    <a href="https://www.kaggle.com/code/varunsaikanuri/life-expectancy-visualization-and-prediction" target="_blank">kernel.</a> Please do check it out and support the author.</p>



## ❗Author's Note:
<div class="alert alert-block alert-info" style="font-size:14px; font-family:verdana;">
    📌 Make sure to run the cells from top to bottom. Also, any suggestions, comments and recommendations to improve the notebook will be highly appreciated. Cheers!
</div>


*** 

# 🏗️Import Necessary Libraries

In [ ]:
# Data Wrangling libraries
import numpy as np
import pandas as pd
import scipy.stats as stats
import datetime

# Visualization Libraries
from IPython.display import display,HTML
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Preprocessing libraries
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# Machine Learning Estimators
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.svm import LinearSVR
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
import xgboost as xgb

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ignore the warnings to remove messy output logs
import warnings
warnings.filterwarnings('ignore')

# 📥Importing the Dataset

In [ ]:
# Importing the dataset
train = pd.read_csv('../input/tabular-playground-series-sep-2022/train.csv')
test = pd.read_csv('../input/tabular-playground-series-sep-2022/test.csv')

# The temp dataset will be used in the EDA and visualization parts.
temp = train.copy()

# 📈Exploratory Data Analysis and Visualization

In [ ]:
# Function to change the date column data type into Timestamp
def change_to_timestamp(data):
    data['date'] = data['date'].apply(lambda x: datetime.datetime.strptime(x, "%Y-%m-%d"))
    return data
    
train = change_to_timestamp(train)
test = change_to_timestamp(test)
temp = change_to_timestamp(temp)

In [ ]:
# Separate the date column into three different columns
def date_feature_extraction(data):
    data['day'] = data.date.apply(lambda x: x.day)
    data['month'] = data.date.apply(lambda x: x.month)
    data['year'] = data.date.apply(lambda x: x.year)
    return data

train = date_feature_extraction(train)
test = date_feature_extraction(test)
temp = date_feature_extraction(temp)

In [ ]:
# Display the train dataset
display(HTML(train.head().to_html()))

In [ ]:
# View Dataset Statistics
display(HTML(train.describe().to_html()))

In [ ]:
# Get a two-line title for our plots
def get_multi_line_title(title:str, subtitle:str):
    return f"{title}<br><sub>{subtitle}</sub><br>"

title = get_multi_line_title("Distribution of target column (num_sold)", "Distirbution of number of books sold per country")
data = temp[temp['num_sold'] != 0]
top_devices = data.groupby('country')['num_sold'].count().sort_values(ascending=False)[:5].index.tolist()
data = data[data['country'].apply(lambda x: x in top_devices)]
fig = px.histogram(data, x="num_sold", color="country", opacity=0.75, template='plotly_dark',)
fig.update_layout(hovermode='x', title=title)
fig.show()

<div style="font-family:verdana; word-spacing:1.5px;">

<p style="font-size:15px; font-family:verdana; line-height: 1.7em">
The distribution of the `num_sold` column seems to be right skewed with some few outliers selling more than 800 books. It is extremely typical for skewed distributions to have one tail that is significantly longer or dragged out compared to the other tail. A distribution is said to be "skewed right" if the tail is to the right. Lower or upper boundaries on the data frequently cause skewed data. In other words, data with a lower bound are frequently skewed right, whereas data with an upper bound are typically biased left. Start-up effects can also cause skewness. For instance, some processes in reliability applications could experience a high rate of first failures, which could result in left skewness. On the other side, a dependability process can have a protracted startup phase where failures are uncommon and the data would be right-skewed. </p><br>

<p>The following recommendations should be followed if the histogram shows that the data set is right-skewed:<br><ol>
       <li> Compute and publish the sample mean, sample median, and sample mode to quantitatively summarize the data.</li>
        <li>Consider a normalizing transformation such as the Box-Cox transformation
        </li>
        <li>Choose the best-fit distribution (rightly skewed) from the
            <ul>
                <li>Weibull family (for the maximum)</li>
                <li>Gamma Family</li>
                <li>Chi-square family</li>
                <li>Lognormal family</li>
                <li>Power lognormal family</li>
            </ul>    
        </li>
    </ol>
</p>

For more in-depth information about this, You can visit this [link](https://www.itl.nist.gov/div898/handbook/eda/section3/eda33e6.htm#:~:text=For%20skewed%20distributions%2C%20it%20is,is%20on%20the%20left%20side.).
</div>

In [ ]:
# Plot the number of books stored based on the store
fig = px.violin(temp,
                x='store',
                y='num_sold',
                color='store',
                template='plotly_dark',
                box=True,
                title='Number of Books sold Based on Store')
fig.show()

<p style="font-size:15px; font-family:verdana; line-height: 1.7em">A violin plot, which depicts data peaks, is a cross between a box plot and a kernel density plot. It is used to show how numerical data is distributed. Violin plots provide summary statistics as well as the density of each variable, unlike box plots, which can only show summary statistics. In the violin plot, there are more outsude points within the Kaggle Mart feature compared to the Kaggle Rama. It can be also noted that the former has a wider Inter Quartile Range (IQR) compared to the latter. </p>

![Violin Plots](https://miro.medium.com/max/780/1*TTMOaNG1o4PgQd-e8LurMg.png)
    
<p style="font-size:15px; font-family:verdana; line-height: 1.7em">
You can read about an in-depth explanation of violin plots in this <a href="https://towardsdatascience.com/violin-plots-explained-fb1d115e023d" target="_blank">link.</a></p>

<div class="alert alert-block alert-info" style="font-size:14px; font-family:verdana;">
    📌 Hover your mouse in the graph to examine the data distributions.
</div>

In [ ]:
# Plot the number of books stored based on the product
fig = px.violin(temp,
                x='product',
                y='num_sold',
                color='product',
                template='plotly_dark',
                box=True,
                title='Number of Books sold Based on Product')
fig.show()

<div class="alert alert-block alert-info" style="font-size:14px; font-family:verdana;">
    📌 Drag the slider or press the play button to see the different graphs per country.
</div>

In [ ]:
# Plot Countrywise number of books sold
fig = px.line(
    temp.sort_values(by='date'),
    x='date',
    y='num_sold',
    animation_frame='country',
    animation_group='date',
    color='country',
    markers=True,
    template='plotly_dark',title='<b> Number of books sold per country over Years')
fig.show()

In [ ]:
# Plot monthly book sales per country
px.scatter(temp,
           y='month',
           x='num_sold',
           color='country',
           size='num_sold',
           template='plotly_dark',
           opacity=0.6,
           title='<b> Monthly number of books sold per country')

In [ ]:
# Daily book sales per country
px.scatter(temp,
           y='day',
           x='num_sold',
           color='country',
           size='num_sold',
           template='plotly_dark',
           opacity=0.6,
           title='<b> Daily number of books sold per country')

In [ ]:
# Monthly book sales per store
px.scatter(temp,
           y='month',
           x='num_sold',
           color='store',
           size='num_sold',
           template='plotly_dark',
           opacity=0.6,
           title='<b> Monthly number of books sold per store')

In [ ]:
# Daily book sales per store
px.scatter(temp,
           y='day',
           x='num_sold',
           color='store',
           size='num_sold',
           template='plotly_dark',
           opacity=0.6,
           title='<b> Daily number of books sold per store')

In [ ]:
# Monthly book sales per product
fig = px.scatter(temp,
           y='month',
           x='num_sold',
           color='product',
           size='num_sold',
           template='plotly_dark',
           opacity=0.6,
           title='<b> Monthly number of books sold per product')


fig.show()

In [ ]:
# Daily book sales per store
fig = px.scatter(temp,
           y='day',
           x='num_sold',
           color='product',
           size='num_sold',
           template='plotly_dark',
           orientation ='h',
           opacity=1.0,
           title='<b> Daily number of books sold per store')
fig.show()

In [ ]:
# Plot the number of books stored based on the store and country
fig = px.histogram(temp,
                x='store',
                y='num_sold',
                color='country',
                template='plotly_dark',
                title='Total Number of Books sold based on Store per Country')
fig.show()

In [ ]:
# Plot the number of books stored based on the product and country
fig = px.histogram(temp,
                x='product',
                y='num_sold',
                color='country',
                template='plotly_dark',
                title='Total Number of Books sold based on Product per Country')
fig.show()

# 📝Data Preprocessing 

<p style="font-size:15px; font-family:verdana; line-height: 1.7em">In this section, We will first check for missing data within our train and test splits. We are then to view different statistics in the data and see if there are any interesting findings. Then, we are going to perform a simple data preprocessing using Label encoder to convert categorical data into numerical format.</p>

In [ ]:
# Check for missing values in train
train.isna().sum()

In [ ]:
# Check column informations
train.info()

<p style="font-size:15px; font-family:verdana; line-height: 1.7em">Since there is no missing in the data, we are going to proceed with encoding the data.</p>

In [ ]:
# Encoding the data using Label Encoder
encoder = LabelEncoder()
def encode_data(data, categories=['country', 'store', 'product']):
    for cat in categories:
        data[cat] = encoder.fit_transform(data[[cat]])
    return data

train = encode_data(train)
test = encode_data(test)
temp = encode_data(temp)

In [ ]:
# View encoded data
train.head()

In [ ]:
# Check variable descriptions
train.describe()

In [ ]:
def plot_heatmap(data, cmap=sns.color_palette(palette='rocket', as_cmap=True), height=3, linewidth=1, title=' ', subtitle=' '):
    sns.set(style = 'whitegrid', rc = {'figure.figsize': (20,height)})

    g = sns.heatmap(data=data, cmap=cmap,
                    linewidths = linewidth,
                    annot=True,
                    fmt='.2f',
                    cbar_kws = dict(
                        location = 'right'))

    g.set_xlabel(' \n\n\n\n')
    g.set_ylabel(' \n\n\n\n')

    g.set_xticklabels([tick_label.get_text().title() for tick_label in g.get_xticklabels()])
    g.set_yticklabels([tick_label.get_text().title() for tick_label in g.get_yticklabels()])

    g.set_title(f'\n\n\n\n{title}\n\n'.upper(),
                loc = 'left',
                fontdict = dict(
                    fontsize = 20,
                    fontweight = 'bold'))
    
    plt.text(s=f'{subtitle}',
             alpha=0.5,
             x = 0,
             y = -0.25,
             horizontalalignment = 'left',
             verticalalignment = 'top',
             fontsize=16)

    plt.text(s = ' ',
             x = 1.2,
             y = 1,
             transform = g.transAxes)

    return g

In [ ]:
sns.set(style = 'whitegrid', rc = {'figure.figsize': (20,15)})
plot_heatmap(
    height=15,
    data = train.corr(),
    title = 'Train Dataset Correlation Overview',
    subtitle = 'Method of correlation: Pearson Correlation Coefficient'
);

<p style="font-size:15px; font-family:verdana; line-height: 1.7em">Now we will split the data into training and validation sets. The training set is where the model will do the fitting and the validation set will be used to evaluate the model. Since we have already feature engineered each values of the `date` column, we will also be dropping it.</p>

In [ ]:
# Splitting the data
X = train.drop(['num_sold', 'date'], axis=1)
y = train.num_sold

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

# 🤹Training with Different Estimators
<div style="font-family:verdana; word-spacing:1.5px;">
    <p style="font-size:15px; font-family:verdana; line-height: 1.7em">In this section, we are going to try to train our training and validation sets with different kinds of machine learning estimators. We are then going to evaluate each estimator with the following metrics:<ol>
    <li><b>Mean Square Error(MSE)/Root Mean Square Error(RMSE)</b></li>
    <p><br>MSE is calculated by the sum of square of prediction error which is real output minus predicted output and then divide by the number of data points. It gives you an absolute number on how much your predicted results deviate from the actual number. You cannot interpret many insights from one single result but it gives you a real number to compare against other model results and help you select the best regression model.<br><img src='https://miro.medium.com/max/852/1*aFBAjR7kzWirbqORnYa43Q.png'><br>Root Mean Square Error(RMSE) is the square root of MSE. It is used more commonly than MSE because firstly sometimes MSE value can be too big to compare easily. Secondly, MSE is calculated by the square of error, and thus square root brings it back to the same level of prediction error and makes it easier for interpretation.</p>
    <li><b>R Square/Adjusted R Square</b></li>
    <p><br>R Square measures how much variability in dependent variable can be explained by the model. It is the square of the Correlation Coefficient(R) and that is why it is called R Square.<br><img src='https://miro.medium.com/max/1050/1*e1n9VlEFgaJWLKyaJQZwlw.png'><br>R Square is a good measure to determine how well the model fits the dependent variables. However, it does not take into consideration of overfitting problem. If your regression model has many independent variables, because the model is too complicated, it may fit very well to the training data but performs badly for testing data. That is why Adjusted R Square is introduced because it will penalize additional independent variables added to the model and adjust the metric to prevent overfitting issues.</p>
    <li><b>Mean Absolute Error(MAE)</b></li>
    <p><br>Mean Absolute Error(MAE) is similar to Mean Square Error(MSE). However, instead of the sum of square of error in MSE, MAE is taking the sum of the absolute value of error.<br><img src='https://miro.medium.com/max/780/1*tu6FSDz_FhQbR3UHQIaZNg.png'><br>Compare to MSE or RMSE, MAE is a more direct representation of sum of error terms. MSE gives larger penalization to big prediction error by square it while MAE treats all errors the same.</p>
    </ol>
    </p>
</div>

In [ ]:
# Function to calculate metric results
def calculate_results(y_true, y_pred):
    # Calculate model Mean Absolute Error (MAE)
    model_mae = mean_absolute_error(y_val, y_pred)
    # Calculate model Mean Squrared Error (MSE)
    model_mse = mean_squared_error(y_val, y_pred)
    # Calculate model Root Mean Squared Error (RMSE)
    model_rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    # Calculate Adjusted R2_score
    model_r2 = r2_score(y_val, y_pred)
    # Calculate Root Mean Squared Log Error
    model_rmsle = np.log(np.sqrt(mean_squared_error(y_val, y_pred)))
    
    
    model_results = {"Mean Absolute Error (MAE)": model_mae,
                     "Mean Squared Error (MSE)": model_mse,
                     "Root Mean Squared Error (RMSE)": model_rmse,
                     "Adjusted R^2 Score": model_r2,
                     "Root Mean Squared Log Error": model_rmsle}
    return model_results



## K Nearest Neighbors (KNN)
<p style="font-size:15px; font-family:verdana; line-height: 1.7em">KNN regression is a non-parametric method that, in an intuitive manner, approximates the association between independent variables and the continuous outcome by averaging the observations in the same neighbourhood.<p>

In [ ]:
# Predict using KNN Regressor
knn_model = KNeighborsRegressor() 
y_pred = knn_model.fit(X_train, y_train).predict(X_val)
    
knn_results = calculate_results(y_val, y_pred)
pd.DataFrame(knn_results ,index=['values']).T

## Random Forest Regressor
<p style="font-size:15px; font-family:verdana; line-height: 1.7em">A random forest regressor. A random forest is a meta estimator that fits a number of classifying decision trees on various sub-samples of the dataset and uses averaging to improve the predictive accuracy and control over-fitting.</p>

In [ ]:
# Predict using Random Forest Regressor
rf_model = RandomForestRegressor() 
y_pred = rf_model.fit(X_train, y_train).predict(X_val)
    
rf_results = calculate_results(y_val, y_pred)
pd.DataFrame(rf_results ,index=['values']).T

## Decision Tree Regressor
<p style="font-size:15px; font-family:verdana; line-height: 1.7em">Decision tree builds regression or classification models in the form of a tree structure. It breaks down a dataset into smaller and smaller subsets while at the same time an associated decision tree is incrementally developed. The final result is a tree with decision nodes and leaf nodes.</p> 

In [ ]:
# Predict using Decision Tree Regressor
dt_model = DecisionTreeRegressor() 
y_pred = dt_model.fit(X_train, y_train).predict(X_val)
    
dt_results = calculate_results(y_val, y_pred)
pd.DataFrame(dt_results ,index=['values']).T

## Linear Regression
<p style="font-size:15px; font-family:verdana; line-height: 1.7em">In statistics, linear regression is a linear approach for modelling the relationship between a scalar response and one or more explanatory variables. The case of one explanatory variable is called simple linear regression; for more than one, the process is called multiple linear regression.</p>

In [ ]:
# Predict using Linear Regression
lr_model = LinearRegression() 
y_pred = lr_model.fit(X_train, y_train).predict(X_val)
    
lr_results = calculate_results(y_val, y_pred)
pd.DataFrame(lr_results ,index=['values']).T

## Extra Trees Regressor
<p style="font-size:15px; font-family:verdana; line-height: 1.7em">An extra-trees regressor. This class implements a meta estimator that fits a number of randomized decision trees (a.k.a. extra-trees) on various sub-samples of the dataset and uses averaging to improve the predictive accuracy and control over-fitting.</p>

In [ ]:
# Predict using Extra Tree Regressor
et_model = ExtraTreesRegressor() 
y_pred = et_model.fit(X_train, y_train).predict(X_val)
    
et_results = calculate_results(y_val, y_pred)
pd.DataFrame(et_results ,index=['values']).T

## Linear Support Vector Regression
<p style="font-size:15px; font-family:verdana; line-height: 1.7em">Based on support vector machines method, the Linear SVR is an algorithm to solve the regression problems. The Linear SVR algorithm applies linear kernel method and it works well with large datasets. L1 or L2 method can be specified as a loss function in this model.</p>

**Note:** Running this model may take a long time depending on the number of iterations. Change `max_iter` to your preference.

In [ ]:
# Predict using Support Vector Regression
svr_model = LinearSVR(max_iter=1000)
y_pred = svr_model.fit(X_train, y_train).predict(X_val)
    
svr_results = calculate_results(y_val, y_pred)
pd.DataFrame(svr_results ,index=['values']).T

## Ridge Regression
<p style="font-size:15px; font-family:verdana; line-height: 1.7em">Ridge regression is a model tuning method that is used to analyse any data that suffers from multicollinearity. This method performs L2 regularization. When the issue of multicollinearity occurs, least-squares are unbiased, and variances are large, this results in predicted values being far away from the actual values.</p>

In [ ]:
# Predict using Ridge Regression
ridge_model = Ridge()
y_pred = ridge_model.fit(X_train, y_train).predict(X_val)
    
ridge_results = calculate_results(y_val, y_pred)
pd.DataFrame(ridge_results ,index=['values']).T

## Lasso Regression
<p style="font-size:15px; font-family:verdana; line-height: 1.7em">Lasso regression is a regularization technique. It is used over regression methods for a more accurate prediction. This model uses shrinkage. Shrinkage is where data values are shrunk towards a central point as the mean. The lasso procedure encourages simple, sparse models (i.e. models with fewer parameters)</p>

In [ ]:
# Predict using Ridge Regression
lasso_model = Lasso()
y_pred = lasso_model.fit(X_train, y_train).predict(X_val)
    
lasso_results = calculate_results(y_val, y_pred)
pd.DataFrame(lasso_results ,index=['values']).T

## XGBoost Regression
<p style="font-size:15px; font-family:verdana; line-height: 1.7em">
Gradient boosting refers to a class of ensemble machine learning algorithms that can be used for classification or regression predictive modeling problems.</p>

<p style="font-size:15px; font-family:verdana; line-height: 1.7em">
Ensembles are constructed from decision tree models. Trees are added one at a time to the ensemble and fit to correct the prediction errors made by prior models. This is a type of ensemble machine learning model referred to as boosting.</p>

<p style="font-size:15px; font-family:verdana; line-height: 1.7em">
Models are fit using any arbitrary differentiable loss function and gradient descent optimization algorithm. This gives the technique its name, “gradient boosting,” as the loss gradient is minimized as the model is fit, much like a neural network.</p>

In [ ]:
# Predict using Ridge Regression
xgb_model = xgb.XGBRegressor()
y_pred = xgb_model.fit(X_train, y_train).predict(X_val)
    
xgb_results = calculate_results(y_val, y_pred)
pd.DataFrame(xgb_results ,index=['values']).T

# 📊All Model Results

In [ ]:
# Combine model results into a DataFrame
all_model_results = pd.DataFrame({"K-Nearest Neighbors": knn_results,
                                  "Linear Regression": lr_results,
                                  "Random Forest Regression": rf_results,
                                  "Decision Tree Regressor": dt_results,
                                  "Extra Trees Regressor": et_results,
                                  "Linear SVR": svr_results,
                                  "Ridge Regression": ridge_results,
                                  "Lasso Regression": lasso_results,
                                  "XGBoost Regression": xgb_results})
all_model_results = all_model_results.T
all_model_results

In [ ]:
plt.figure(figsize=(20,10))
metric = 'Mean Absolute Error (MAE)'
g = all_model_results[metric].plot(kind='bar')
g.set_title(f'\n\n\n\n"{metric}"\n\n'.upper(),
                loc = 'center',
                fontdict = dict(
                    fontsize = 20,
                    fontweight = 'bold'));

In [ ]:
plt.figure(figsize=(20,10))
metric = 'Mean Squared Error (MSE)'
g = all_model_results[metric].plot(kind='bar')
g.set_title(f'\n\n\n\n"{metric}"\n\n'.upper(),
                loc = 'center',
                fontdict = dict(
                    fontsize = 20,
                    fontweight = 'bold'));

In [ ]:
plt.figure(figsize=(20,10))
metric = 'Root Mean Squared Error (RMSE)'
g = all_model_results[metric].plot(kind='bar')
g.set_title(f'\n\n\n\n"{metric}"\n\n'.upper(),
                loc = 'center',
                fontdict = dict(
                    fontsize = 20,
                    fontweight = 'bold'));

In [ ]:
plt.figure(figsize=(20,10))
metric = 'Adjusted R^2 Score'
g = all_model_results[metric].plot(kind='bar')
g.set_title(f'\n\n\n\n"{metric}"\n\n'.upper(),
                loc = 'center',
                fontdict = dict(
                    fontsize = 20,
                    fontweight = 'bold'));

In [ ]:
plt.figure(figsize=(20,10))
metric = 'Root Mean Squared Log Error'
g = all_model_results[metric].plot(kind='bar')
g.set_title(f'\n\n\n\n"{metric}"\n\n'.upper(),
                loc = 'center',
                fontdict = dict(
                    fontsize = 20,
                    fontweight = 'bold'));

***

<div style="color:white;
           display:fill;
           border-radius:5px;
           background-color:#5642C5;
           font-size:110%;
           font-family:Verdana;
           letter-spacing:0.5px">
        <p style="padding: 10px;
              color:white;">
            Thanks for viewing my work. If you like it, consider sharing it to others or give feedback to improve the notebook. Have a beautiful day my friend.
        </p>
    </div>

<center><img src='https://media4.giphy.com/media/M9gbBd9nbDrOTu1Mqx/giphy.gif?cid=790b7611704aa2ca4e403287801480a6c753abf45f3e6242&rid=giphy.gif&ct=s' 
     height=30px width=160px /></center>